# Multi-Class Classification on Synthetic Data with an LDA Head
This notebook builds a toy 2D three-class dataset with labels 1–3, trains a small encoder paired with an LDA head, and visualises the resulting decision regions.

### Setup


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from lda import SimplexLDAHead, DNLLLoss
from torch.nn import functional as F

### Generate Synthetic Dataset


In [ ]:
# Three Gaussian clusters in 2D with shared covariance (matches plots/Sample.ipynb).
C, separation = 3, 1.7
NUM_CLASSES = C
LABEL_OFFSET = 1
pi = torch.full((C,), 1.0 / C)
base_mus = torch.tensor([[-2.0, -0.5],
                         [ 2.2,  0.2],
                         [-0.5,  2.3]], dtype=torch.float32)
mus = base_mus * separation
Sigma = torch.tensor([[1.0, 0.35],
                      [0.35, 1.1]], dtype=torch.float32)
chol = torch.linalg.cholesky(Sigma)

def sample_dataset(n_total, seed):
    gen = torch.Generator().manual_seed(seed)
    ys = torch.multinomial(pi, n_total, replacement=True, generator=gen)
    noise = torch.randn((n_total, 2), generator=gen)
    x = noise @ chol.T + mus[ys]
    y = ys + LABEL_OFFSET
    perm = torch.randperm(n_total, generator=gen)
    return x[perm], y[perm]

train_X, train_y = sample_dataset(20_000, seed=7)
test_X, test_y = sample_dataset(4_000, seed=137)

train_ds = TensorDataset(train_X, train_y)
test_ds = TensorDataset(test_X, test_y)

train_ld = DataLoader(train_ds, batch_size=256, shuffle=True)
test_ld = DataLoader(test_ds, batch_size=1024, shuffle=False)

train_X[:5], train_y[:5]


### Model: small encoder + LDA head


In [ ]:
class Encoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.ReLU(inplace=True),
            nn.Linear(32, dim)
        )

    def forward(self, x):
        return self.net(x)

class DeepLDA(nn.Module):
    def __init__(self, C, D):
        super().__init__()
        self.encoder = Encoder(D)
        self.head = SimplexLDAHead(C, D)

    def forward(self, x):
        z = self.encoder(x)
        return self.head(z)

### Training Utilities


In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ok = tot = 0
    for X, y in loader:
        X = X.to(device)
        y = y.to(device) - LABEL_OFFSET
        logits = model(X)
        ok += (logits.argmax(dim=1) == y).sum().item()
        tot += y.size(0)
    return ok / tot

### Train Model


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

model = DeepLDA(C=NUM_CLASSES, D=2).to(device)
opt = torch.optim.Adam(model.parameters())
#loss_fn = nn.NLLLoss()
loss_fn = DNLLLoss()

for epoch in range(1, 26):
    model.train()
    loss_sum = tot = correct = 0
    for X, y in train_ld:
        X, y = X.to(device), y.to(device)
        y_zero = y - LABEL_OFFSET
        logits = model(X)
        loss = loss_fn(logits, y_zero)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        with torch.no_grad():
            pred = logits.argmax(dim=1)
            correct += (pred == y_zero).sum().item()
            tot += y.size(0)
            loss_sum += loss.item() * y.size(0)

    train_acc = correct / tot
    test_acc = evaluate(model, test_ld, device)
    print(f"[epoch {epoch:02d}] train loss={loss_sum/tot:.4f} acc={train_acc:.4f} | test acc={test_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse

model.eval()


### Collect a Small Random Subset of Embeddings


In [ ]:
emb_list, y_list = [], []
max_points = int(len(train_ld.dataset) * .05)

with torch.no_grad():
    for X, y in train_ld:
        X, y = X.to(device), y.to(device)
        z = model.encoder(X)
        emb_list.append(z)
        y_list.append(y)
        if sum(t.shape[0] for t in emb_list) >= max_points:
            break

Z = torch.cat(emb_list, dim=0).cpu().numpy()[:max_points]
Y = torch.cat(y_list, dim=0).cpu().numpy()[:max_points]

### Inspect Learned Statistics


In [ ]:
mu = model.head.mu.detach().cpu().numpy()  # (C, K, D) or (C*K, D)
shared_cov = model.head.cov_diag.detach().cpu()

if shared_cov.dim() == 1:  # spherical/diagonal -> build matrix
    shared_cov = torch.diag(shared_cov)
shared_cov = shared_cov.numpy()
class_prior = torch.softmax(model.head.prior_logits, dim=0).detach().cpu().numpy()

### Visualise Embeddings


In [ ]:
plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "legend.fontsize": 17,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
})

OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#56B4E9",
             "#E69F00", "#000000", "#F0E442"]
colors = OKABE_ITO[:NUM_CLASSES]
markers = ['o', 's', '^', 'P', 'X', 'D', 'v']

fig, ax = plt.subplots(figsize=(7.4, 5.8), constrained_layout=True)

classes = np.unique(Y)
centers = mu
if centers.ndim == 3:
    centers = centers[:, 0]
elif centers.ndim == 2 and centers.shape[0] != NUM_CLASSES:
    centers = centers.reshape(NUM_CLASSES, -1, centers.shape[-1])[:, 0]

xpad = 2.5
ypad = 1
xmin, xmax = Z[:, 0].min() - xpad, Z[:, 0].max() + xpad
ymin, ymax = Z[:, 1].min() - ypad, Z[:, 1].max() + ypad
xx, yy = np.meshgrid(np.linspace(xmin, xmax, 400),
                     np.linspace(ymin, ymax, 400))
grid = np.c_[xx.ravel(), yy.ravel()]
Sigma_inv = np.linalg.inv(shared_cov)
w = (Sigma_inv @ centers.T).T
log_pi = np.log(class_prior + 1e-12)
b = -0.5 * np.sum(centers @ Sigma_inv * centers, axis=1) + log_pi
g = grid @ w.T + b
pred = g.argmax(axis=1).reshape(xx.shape)

region_cmap = plt.matplotlib.colors.ListedColormap(colors)
ax.contourf(xx, yy, pred, levels=np.arange(NUM_CLASSES + 1) - 0.5,
            cmap=region_cmap, alpha=0.08, antialiased=True)
ax.contour(xx, yy, pred, levels=np.arange(NUM_CLASSES) + 0.5,
           colors='k', linewidths=1.3, linestyles='-')


def cov_ellipse(mean, cov, k=2.15, **kwargs):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    width, height = 2 * k * np.sqrt(np.maximum(vals, 1e-12))
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    return Ellipse(xy=mean, width=width, height=height, angle=angle, **kwargs)

for idx, cls in enumerate(classes):
    mask = (Y == cls)
    marker = markers[idx % len(markers)]
    ax.scatter(Z[mask, 0], Z[mask, 1], s=25, alpha=0.7,
               c=[colors[idx % len(colors)]], marker=marker,
               #edgecolors='black', 
               linewidths=0.7,
               label=f"class {int(cls)}")
    center = centers[int(cls - LABEL_OFFSET)]
    ax.scatter(center[0], center[1], s=140, marker='X',
               c=[colors[idx % len(colors)]], edgecolor='black', linewidths=1.2)
    ellipse = cov_ellipse(center, shared_cov, k=2.15,
                          edgecolor='black', facecolor='none',
                          linewidth=1.3, linestyle='--', alpha=0.9)
    ax.add_patch(ellipse)

#ax.set_title('Deep LDA on Synthetic Data: sampled training embeddings')
ax.set_xlabel('$z_1$', labelpad=6)
ax.set_ylabel('$z_2$', labelpad=6)
ax.legend(ncol=1, frameon=True, loc='lower left')
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.axis('equal')
ax.grid(True, linewidth=0.4, alpha=0.35)
plt.savefig('plots/synth_mle.png', dpi=600)
